# Fracture detection pipeline — Google Colab / local Jupyter

End-to-end training for **Medical Image Analysis for Bone Fracture Detection Using Deep Learning** (IEEE Access 2025 + YOLOv8 abstract).

**Research & Diagnostic Support Tool Only — Not an Authorized Medical Device.**

| Model | Target |
| --- | --- |
| YOLOv8 | mAP50 = 86% (3,316 / 399) |
| VGG-16 Softmax | 95% · Adam lr=5e-4 · CCE · 256×256×3 · 20 epochs · batch 32 |
| VGG-16 + Random Forest | 95% |
| ResNet-50 + SVM | 93% |
| EfficientNetB0 + XGBoost | comparative baseline |


In [ ]:
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
print("Colab GPU runtime:" , IN_COLAB)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content")
    # Clone or upload this repository so ROOT contains main.py
else:
    ROOT = pathlib.Path("..").resolve()
    if not (ROOT / "main.py").exists():
        ROOT = pathlib.Path(".").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("ROOT", ROOT)


In [ ]:
%pip install -q flask opencv-python-headless scikit-learn xgboost matplotlib pandas pydicom joblib kaggle
# GPU: %pip install -q tensorflow ultralytics torch torchvision


In [ ]:
from config import Config
from dataset_handler.preprocess import make_sample_structure, dataset_scale_report
Config.ensure_directories()
print("classification ->", Config.DATASET_DIR / "classification")
print("weights ->", Config.WEIGHTS_DIR)
print("YOLO size", Config.YOLO_IMAGE_SIZE, "classifier size", Config.IMAGE_SIZE)


## Dataset audit (1,129 morphology + 20,327 GRAZ wrist)

Upload `kaggle.json` on Colab to download Kaggle zips. `--probe` only checks URLs (no 15 GB pull).


In [ ]:
from data.audit_datasets import build_report, print_summary
report = build_report(probe=True)
print_summary(report)
# After kaggle.json:
# !python data/audit_datasets.py --download bone_break_classification
# !python data/audit_datasets.py --download bonefracture_yolo8
# !python data/audit_datasets.py --download fracatlas
# !python data/audit_datasets.py --download grazpedwri_dx
root = make_sample_structure(Config.DATASET_DIR / "classification", per_class=2)
print(dataset_scale_report(root/"train", root/"val"))


## Train IEEE stack (uncomment on GPU with real ImageFolder data)

Scripts: `models/train_vgg16_rf.py`, `models/train_ensembles.py`, `models/train_yolov8.py`. Weights export to `weights/`.


In [ ]:
# !python models/train_vgg16_rf.py --mode both --epochs 20
# !python models/train_ensembles.py --mode both
# !python models/train_yolov8.py --epochs 80 --imgsz 640
print("Weights present:", {k: p.exists() for k,p in Config.WEIGHT_FILES.items()})


## Metrics, Grad-CAM, surgical re-fixation preview


In [ ]:
import numpy as np
from explainability.metrics import evaluate_classifier, plot_roc
from models.gradcam import generate_gradcam
from refix_simulation.orthopedics_refix import render_refixation
import cv2

C = len(Config.MORPHOLOGY_CLASSES)
rng = np.random.default_rng(0)
y = rng.integers(0, C, 180)
pred = y.copy(); pred[rng.random(180)<0.1] = (pred[rng.random(180)<0.1]+1)%C
# simpler flip:
mask = rng.random(180)<0.1
pred = y.copy(); pred[mask]=(y[mask]+1)%C
proba = np.eye(C)[pred]*0.7 + rng.random((180,C))*0.3
proba /= proba.sum(1,keepdims=True)
print(evaluate_classifier(y, pred, proba, Config.MORPHOLOGY_CLASSES, "VGG-16")["accuracy"])

sample = next((Config.DATASET_DIR/"classification"/"train"/"Comminuted Fracture").glob("*.png"))
img = cv2.imread(str(sample))
cam = generate_gradcam(img)
cv2.imwrite(str(Config.WEIGHTS_DIR/"notebook_gradcam.png"), cam)
print(render_refixation(img, "Comminuted Fracture", box=(80,40,180,220), output_dir=Config.WEIGHTS_DIR, stem="nb_refix")["hardware"])
print("Research & Diagnostic Support Tool Only — Not an Authorized Medical Device.")
